In [21]:
import os
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [38]:
data_df = pd.read_csv(r"./hourly_cleaned_power_data.csv", sep=',')
data_df

,datetime,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Other_energy_wh,hour,day_of_week,month,is_weekend
0,2006-12-16 17:00:00,4.222889,0.229000,234.643889,18.100000,0.0,0.527778,16.861111,52.992593,17,Saturday,December,True
1,2006-12-16 18:00:00,3.632200,0.080033,234.580167,15.600000,0.0,6.716667,16.866667,36.953333,18,Saturday,December,True
2,2006-12-16 19:00:00,3.400233,0.085233,233.232500,14.503333,0.0,1.433333,16.683333,38.553889,19,Saturday,December,True
3,2006-12-16 20:00:00,3.268567,0.075100,234.071500,13.916667,0.0,0.000000,16.783333,37.692778,20,Saturday,December,True
4,2006-12-16 21:00:00,3.056467,0.076667,237.158667,13.046667,0.0,0.416667,17.216667,33.307778,21,Saturday,December,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
34584,2010-11-26 17:00:00,1.725900,0.061400,237.069667,7.216667,0.0,0.000000,12.866667,15.898333,17,Friday,November,False
34585,2010-11-26 18:00:00,1.573467,0.053700,237.531833,6.620000,0.0,0.000000,0.000000,26.224444,18,Friday,November,False
34586,2010-11-26 19:00:00,1.659333,0.060033,236.741000,7.056667,0.0,0.066667,0.000000,27.588889,19,Friday,November,False
34587,2010-11-26 20:00:00,1.163700,0.061167,239.396000,4.913333,0.0,1.066667,0.000000,18.328333,20,Friday,November,False



    - Original Minute-by-Minute Data: The dataset covers a span of roughly 4 years (Dec 2006 to Nov 2010).
    - The Calculation: 4 years × 365 days × 24 hours = ~35,040 hours.
    - The 34,589 rows you see in the hourly dataset represents every single hour in that period that contained data.


In [40]:
dup_df = data_df.copy()

In [24]:
dup_df.isnull().sum()

,0
datetime,0
Global_active_power,0
Global_reactive_power,0
Voltage,0
Global_intensity,0
Sub_metering_1,0
Sub_metering_2,0
Sub_metering_3,0
Other_energy_wh,0
hour,0


In [41]:
dup_df.dtypes

,0
datetime,object
Global_active_power,float64
Global_reactive_power,float64
Voltage,float64
Global_intensity,float64
Sub_metering_1,float64
Sub_metering_2,float64
Sub_metering_3,float64
Other_energy_wh,float64
hour,int64


In [42]:
from sklearn.impute import SimpleImputer

The impute object you see is an instance of SimpleImputer from the sklearn.impute library. Its purpose is to handle missing values in a dataset.

1. What is an Imputer?
- An imputer is a data preprocessing tool used to fill in (or 'impute') missing values in your dataset.

- Missing values can cause issues with many machine learning models, so handling them is an important step.

2. What is SimpleImputer?
- This specific imputer replaces missing values using a simple strategy.

- Here, `strategy='median'` means that any missing values in a column will be replaced by the median value of that column.

- Other strategies include mean (replace with the mean) or most_frequent (replace with the most frequent value).

In [43]:
num_cols = dup_df.select_dtypes(include=['int64', 'float64']).columns
impute = SimpleImputer(strategy='median')
dup_df[num_cols] = impute.fit_transform(dup_df[num_cols])
print(num_cols)

Index(['Global_active_power', 'Global_reactive_power', 'Voltage',
       'Global_intensity', 'Sub_metering_1', 'Sub_metering_2',
       'Sub_metering_3', 'Other_energy_wh', 'hour'],
      dtype='object')


In [44]:
cat_cols = dup_df.select_dtypes(exclude=['int64', 'float64']).columns
impute = SimpleImputer(strategy='most_frequent')
dup_df[cat_cols] = impute.fit_transform(dup_df[cat_cols])
print(cat_cols)

Index(['datetime', 'day_of_week', 'month', 'is_weekend'], dtype='object')


### 1. Convert 'datetime' to datetime objects and set as index

This is a crucial first step for proper time-series analysis in pandas.

In [45]:
# Convert 'datetime' column to datetime objects
dup_df['datetime'] = pd.to_datetime(dup_df['datetime'])

# Set 'datetime' as the DataFrame index
dup_df = dup_df.set_index('datetime')

# Display the updated DataFrame info to confirm changes
display(dup_df.info())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 34589 entries, 2006-12-16 17:00:00 to 2010-11-26 21:00:00
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Global_active_power    34589 non-null  float64
 1   Global_reactive_power  34589 non-null  float64
 2   Voltage                34589 non-null  float64
 3   Global_intensity       34589 non-null  float64
 4   Sub_metering_1         34589 non-null  float64
 5   Sub_metering_2         34589 non-null  float64
 6   Sub_metering_3         34589 non-null  float64
 7   Other_energy_wh        34589 non-null  float64
 8   hour                   34589 non-null  float64
 9   day_of_week            34589 non-null  object 
 10  month                  34589 non-null  object 
 11  is_weekend             34589 non-null  object 
dtypes: float64(9), object(3)
memory usage: 3.4+ MB


None

### 2. Create Lag Features

Lag features introduce past information into the current observation. I'll create lags for `Global_active_power`.

In [46]:
# Create lag features for 'Global_active_power'
dup_df['Global_active_power_lag1'] = dup_df['Global_active_power'].shift(1)  # 1 hour lag
dup_df['Global_active_power_lag24'] = dup_df['Global_active_power'].shift(24) # Daily lag
dup_df['Global_active_power_lag168'] = dup_df['Global_active_power'].shift(168) # 1 week lag

display(dup_df[['Global_active_power', 'Global_active_power_lag1',
                'Global_active_power_lag24', 'Global_active_power_lag168']])

,Global_active_power,Global_active_power_lag1,Global_active_power_lag24,Global_active_power_lag168
datetime,,,,
2006-12-16 17:00:00,4.222889,NaN,NaN,NaN
2006-12-16 18:00:00,3.632200,4.222889,NaN,NaN
2006-12-16 19:00:00,3.400233,3.632200,NaN,NaN
2006-12-16 20:00:00,3.268567,3.400233,NaN,NaN
2006-12-16 21:00:00,3.056467,3.268567,NaN,NaN
...,...,...,...,...
2010-11-26 17:00:00,1.725900,1.067933,1.480100,0.824267
2010-11-26 18:00:00,1.573467,1.725900,2.211600,1.018467
2010-11-26 19:00:00,1.659333,1.573467,2.330467,1.249767


### 3. Create Rolling Window Statistics

Rolling means can help smooth out short-term fluctuations and highlight longer-term trends. I'll create a 24-hour (daily) rolling mean for `Global_active_power`.

In [47]:
# Create rolling mean feature for 'Global_active_power' over a 24-hour window
dup_df['Global_active_power_rolling_mean24'] = dup_df['Global_active_power'].rolling(window=24).mean()

# Display the DataFrame with the new rolling mean feature
display(dup_df[['Global_active_power', 'Global_active_power_rolling_mean24']])

,Global_active_power,Global_active_power_rolling_mean24
datetime,,
2006-12-16 17:00:00,4.222889,NaN
2006-12-16 18:00:00,3.632200,NaN
2006-12-16 19:00:00,3.400233,NaN
2006-12-16 20:00:00,3.268567,NaN
2006-12-16 21:00:00,3.056467,NaN
...,...,...
2010-11-26 17:00:00,1.725900,1.253996
2010-11-26 18:00:00,1.573467,1.227407
2010-11-26 19:00:00,1.659333,1.199443


### 4. Feature Engineering for Other Key Variables

I will now extend the lag and rolling window features to `Voltage` and `Global_intensity` to capture additional temporal dependencies.

In [48]:
# Create lag features for Voltage and Global_intensity
dup_df['Voltage_lag1'] = dup_df['Voltage'].shift(1)
dup_df['Global_intensity_lag1'] = dup_df['Global_intensity'].shift(1)

# Create 24-hour rolling mean for Global_intensity
dup_df['Global_intensity_rolling_mean24'] = dup_df['Global_intensity'].rolling(window=24).mean()

# Drop rows with NaN values created by shifting/rolling
dup_df.dropna(inplace=True)

# Display a summary of the new features
display(dup_df[['Voltage', 'Voltage_lag1', 'Global_intensity', 'Global_intensity_rolling_mean24']].head())
print(f"New shape of dataframe: {dup_df.shape}")

,Voltage,Voltage_lag1,Global_intensity,Global_intensity_rolling_mean24
datetime,,,,
2006-12-23 17:00:00,233.644167,240.476833,23.360000,13.149722
2006-12-23 18:00:00,238.000500,233.644167,16.363333,13.356944
2006-12-23 19:00:00,238.729333,238.000500,17.300000,13.388611
2006-12-23 20:00:00,238.518833,238.729333,17.596667,13.494722
2006-12-23 21:00:00,238.594667,238.518833,13.893333,13.263056


New shape of dataframe: (34421, 19)


### 5. Cyclical Features

To capture the cyclical nature of time-based features (e.g., hour, day of week, month), we can transform them using sine and cosine functions. This helps the model understand that, for example, 23:00 is closer to 00:00 than to 12:00.

In [49]:
# Convert 'hour' into cyclical features
dup_df['hour_sin'] = np.sin(2 * np.pi * dup_df['hour'] / 24)
dup_df['hour_cos'] = np.cos(2 * np.pi * dup_df['hour'] / 24)

# Convert 'day_of_week' into cyclical features (assuming 0=Monday, 6=Sunday)
# Note: 'day_of_week' is currently an object. First, convert it to a numerical representation.
dup_df['day_of_week_num'] = pd.Categorical(dup_df['day_of_week'], categories=['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'], ordered=True).codes
dup_df['day_of_week_sin'] = np.sin(2 * np.pi * dup_df['day_of_week_num'] / 7)
dup_df['day_of_week_cos'] = np.cos(2 * np.pi * dup_df['day_of_week_num'] / 7)

# Convert 'month' into cyclical features (assuming 1=January, 12=December)
# Note: 'month' is currently an object. First, convert it to a numerical representation.
dup_df['month_num'] = pd.Categorical(dup_df['month'], categories=['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December'], ordered=True).codes + 1
dup_df['month_sin'] = np.sin(2 * np.pi * dup_df['month_num'] / 12)
dup_df['month_cos'] = np.cos(2 * np.pi * dup_df['month_num'] / 12)

# Drop the original 'hour', 'day_of_week_num', and 'month_num' if you prefer to only use cyclical features
dup_df.drop(columns=['hour', 'day_of_week', 'day_of_week_num', 'month', 'month_num'], inplace=True)

display(dup_df[['hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'month_sin', 'month_cos']].head())
print(f"New shape of dataframe: {dup_df.shape}")

,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos,month_sin,month_cos
datetime,,,,,,
2006-12-23 17:00:00,-0.965926,-2.588190e-01,-0.974928,-0.222521,-2.449294e-16,1.0
2006-12-23 18:00:00,-1.000000,-1.836970e-16,-0.974928,-0.222521,-2.449294e-16,1.0
2006-12-23 19:00:00,-0.965926,2.588190e-01,-0.974928,-0.222521,-2.449294e-16,1.0
2006-12-23 20:00:00,-0.866025,5.000000e-01,-0.974928,-0.222521,-2.449294e-16,1.0
2006-12-23 21:00:00,-0.707107,7.071068e-01,-0.974928,-0.222521,-2.449294e-16,1.0


New shape of dataframe: (34421, 22)


### 6. Rolling Standard Deviation

Beyond rolling means, rolling standard deviation can provide insights into the volatility or variability of a series over a given window. This can be particularly useful for capturing changes in consumption stability.

In [50]:
import numpy as np

# Create a 24-hour rolling standard deviation for 'Global_active_power'
dup_df['Global_active_power_rolling_std24'] = dup_df['Global_active_power'].rolling(window=24).std()

# Drop rows with NaN values created by rolling standard deviation
dup_df.dropna(inplace=True)

display(dup_df[['Global_active_power', 'Global_active_power_rolling_std24']].head())
print(f"New shape of dataframe: {dup_df.shape}")

,Global_active_power,Global_active_power_rolling_std24
datetime,,
2006-12-24 16:00:00,2.007133,1.282565
2006-12-24 17:00:00,1.686500,1.178259
2006-12-24 18:00:00,0.505200,1.232380
2006-12-24 19:00:00,0.454033,1.258524
2006-12-24 20:00:00,0.474100,1.259679


New shape of dataframe: (34398, 23)


In [53]:
dup_df = pd.get_dummies(dup_df, drop_first=True)
dup_df

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Other_energy_wh,Global_active_power_lag1,Global_active_power_lag24,...,Global_intensity_lag1,Global_intensity_rolling_mean24,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos,month_sin,month_cos,Global_active_power_rolling_std24,is_weekend_True
datetime,,,,,,,,,,,,,,,,,,,,,
2006-12-24 16:00:00,2.007133,0.126100,243.880667,8.313333,6.866667,0.866667,9.650000,16.068889,3.096833,4.349100,...,12.823333,12.269167,-0.866025,-5.000000e-01,-0.781831,0.623490,-2.449294e-16,1.000000,1.282565,True
2006-12-24 17:00:00,1.686500,0.097833,238.020333,7.130000,0.000000,0.516667,0.000000,27.591667,2.007133,5.452533,...,8.313333,11.592917,-0.965926,-2.588190e-01,-0.781831,0.623490,-2.449294e-16,1.000000,1.178259,True
2006-12-24 18:00:00,0.505200,0.119400,241.156333,2.110000,0.000000,0.600000,0.000000,7.820000,1.686500,3.879400,...,7.130000,10.999028,-1.000000,-1.836970e-16,-0.781831,0.623490,-2.449294e-16,1.000000,1.232380,True
2006-12-24 19:00:00,0.454033,0.057233,241.419833,1.850000,0.000000,0.000000,0.000000,7.567222,0.505200,4.117833,...,2.110000,10.355278,-0.965926,2.588190e-01,-0.781831,0.623490,-2.449294e-16,1.000000,1.258524,True
2006-12-24 20:00:00,0.474100,0.075367,241.942333,1.956667,0.000000,0.000000,0.000000,7.901667,0.454033,4.181400,...,1.850000,9.703611,-0.866025,5.000000e-01,-0.781831,0.623490,-2.449294e-16,1.000000,1.259679,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2010-11-26 17:00:00,1.725900,0.061400,237.069667,7.216667,0.000000,0.000000,12.866667,15.898333,1.067933,1.480100,...,4.610000,5.257500,-0.965926,-2.588190e-01,-0.433884,-0.900969,-5.000000e-01,0.866025,0.726796,False
2010-11-26 18:00:00,1.573467,0.053700,237.531833,6.620000,0.000000,0.000000,0.000000,26.224444,1.725900,2.211600,...,7.216667,5.144028,-1.000000,-1.836970e-16,-0.433884,-0.900969,-5.000000e-01,0.866025,0.701471,False
2010-11-26 19:00:00,1.659333,0.060033,236.741000,7.056667,0.000000,0.066667,0.000000,27.588889,1.573467,2.330467,...,6.620000,5.029722,-0.965926,2.588190e-01,-0.433884,-0.900969,-5.000000e-01,0.866025,0.668173,False


In [54]:
dup_df.columns.to_list()

['Global_active_power',
 'Global_reactive_power',
 'Voltage',
 'Global_intensity',
 'Sub_metering_1',
 'Sub_metering_2',
 'Sub_metering_3',
 'Other_energy_wh',
 'Global_active_power_lag1',
 'Global_active_power_lag24',
 'Global_active_power_lag168',
 'Global_active_power_rolling_mean24',
 'Voltage_lag1',
 'Global_intensity_lag1',
 'Global_intensity_rolling_mean24',
 'hour_sin',
 'hour_cos',
 'day_of_week_sin',
 'day_of_week_cos',
 'month_sin',
 'month_cos',
 'Global_active_power_rolling_std24',
 'is_weekend_True']

In [55]:
dup_df.to_csv("./processed_household_power_consumption.csv", index=False)